## Import Libraries

In [36]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif



## Load the Dataset

In [37]:
df = pd.read_csv("../data/clean_data.csv")
df.head()

,match_result,home_club,away_club,fixture_date,competition,competition_code,cup_game,home_manager_id,away_manager_id,home_club_recent_fixture_date_1,...,away_club_recent_competition_code_1,away_club_recent_competition_code_2,away_club_recent_competition_code_3,away_club_recent_competition_code_4,away_club_recent_competition_code_5,away_club_recent_competition_code_6,away_club_recent_competition_code_7,away_club_recent_competition_code_8,away_club_recent_competition_code_9,away_club_recent_competition_code_10
0,away,Newell's Old Boys,River Plate,2019-12-01 00:45:00,Superliga,636,False,468196.0,468200.0,2019-11-26 00:10:00,...,1122.0,642.0,636.0,636.0,636.0,1122.0,636.0,642.0,636.0,1122.0
1,home,Real Estelí,Deportivo Las Sabanas,2019-12-01 01:00:00,Primera Division,752,False,516788.0,22169161.0,2019-11-27 21:00:00,...,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0
2,draw,UPNFM,Marathón,2019-12-01 01:00:00,Liga Nacional,734,False,2510608.0,456313.0,2019-11-28 01:15:00,...,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0
3,away,León,Morelia,2019-12-01 01:00:00,Liga MX,743,False,1552508.0,465797.0,2019-11-28 01:00:00,...,743.0,743.0,743.0,743.0,743.0,743.0,743.0,743.0,746.0,743.0
4,home,Cobán Imperial,Iztapa,2019-12-01 01:00:00,Liga Nacional,705,False,429958.0,426870.0,2019-11-27 18:00:00,...,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0


## Time-Based Feature Extraction
The `fixture_date` column is used to extract time-based features such as year, month, and day of the week. 

Before extracting these features, the dates are standardized to UTC. Crucially, the dataset is then sorted chronologically. This completely prevents future data leakage when training our models, ensuring that past data strictly predicts future data.

In [38]:
# Convert to datetime and standardize to UTC
df['fixture_date'] = pd.to_datetime(df['fixture_date'], utc=True)

# Sort chronologically to prevent data leakage
df = df.sort_values('fixture_date').reset_index(drop=True)

# Extract time-based features
df['fixture_year'] = df['fixture_date'].dt.year
df['fixture_month'] = df['fixture_date'].dt.month
df['fixture_day_of_week'] = df['fixture_date'].dt.dayofweek

# Create a season indicator (differentiating summer break months)
df['is_summer_break'] = df['fixture_month'].isin([6, 7]).astype(int)

## Schedule Congestion and Fatigue Indicators
Machine learning models cannot process raw timestamp strings. Furthermore, the absolute date of a previous match is less important than the relative time elapsed. 

Here, we convert the 10 historical fixture dates into a "days since match" numerical feature for both teams. This mathematically represents schedule congestion and potential team fatigue. Afterward, the original raw string columns are dropped to keep the dataset model-ready.

In [39]:
def calculate_rest_days(df, team_type):
    for i in range(1, 11):
        hist_date_col = f"{team_type}_club_recent_fixture_date_{i}"
        rest_days_col = f"{team_type}_days_since_match_{i}"
        
        df[hist_date_col] = pd.to_datetime(df[hist_date_col], errors='coerce', utc=True)
        
        df[rest_days_col] = (df['fixture_date'] - df[hist_date_col]).dt.days
        
        # If a team hasn't played 10 previous matches, fill the missing rest days with a logical default (90 days)
        df[rest_days_col] = df[rest_days_col].fillna(90)
        
        # Drop the original unreadable string column
        df.drop(columns=[hist_date_col], inplace=True)
        
    return df

# Apply the fatigue calculation to both home and away teams
df = calculate_rest_days(df, "home")
df = calculate_rest_days(df, "away")

df.drop(columns=['fixture_date'], inplace=True)

## Match Context Differentiation
Cup matches often have different patterns than regular competition matches. We need to ensure the `cup_game` flag is properly transformed into a strict numerical format.

In [40]:
df['cup_game'] = df['cup_game'].astype(int)

## Historical Form and Performance Indicators
To effectively describe recent club performance and consistency, we calculate aggregate statistics from the last 10 matches. We will engineer the mean goals scored, mean goals conceded, and mean team ratings over the recent rolling window for both the home and away clubs.

In [41]:
def calculate_recent_stats(df, team_type):
    goals_scored_cols = [f"{team_type}_club_recent_goals_scored_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_goals_scored'] = df[goals_scored_cols].mean(axis=1)

    goals_conceded_cols = [f"{team_type}_club_recent_goals_conceded_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_goals_conceded'] = df[goals_conceded_cols].mean(axis=1)

    rating_cols = [f"{team_type}_club_recent_team_rating_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_team_rating'] = df[rating_cols].mean(axis=1)

    return df

# Apply aggregations to both home and away teams
df = calculate_recent_stats(df, "home")
df = calculate_recent_stats(df, "away")

## Categorical Data Transformation
Categorical attributes such as club names and competition names must be transformed into a numerical format for the machine learning algorithms. We apply Label Encoding to these features, as well as to our 3-class target variable (`match_result`). Manager IDs are also strictly cast to numerical floats.

In [42]:
le = LabelEncoder()

# Categorical columns to encode
cat_cols = ['home_club', 'away_club', 'competition']

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Encode target variable (match_result: home, away, draw) into 0, 1, 2
df['match_result'] = le.fit_transform(df['match_result'].astype(str))

# Ensure manager IDs are numerical
df['home_manager_id'] = df['home_manager_id'].astype(float)
df['away_manager_id'] = df['away_manager_id'].astype(float)

## Feature Selection and Additional Feature Engineering

After creating aggregate features from the recent match history, many of the original match-by-match columns become redundant. Keeping both the raw historical features and their aggregated versions may increase the dimensionality of the dataset without providing additional predictive information.

Therefore, redundant columns are removed, and several new features are generated to better capture the relative strength and recent performance of both teams.

The following features are created:

- Home goal difference
- Away goal difference
- Team rating difference
- Goal difference between the two teams
- Average rest days
- Recent home-match ratio
- Recent cup-match ratio

Finally, the original match-by-match goal, rating, and rest-day columns are removed because their information is already summarized by the engineered features.

In [43]:
# Goal difference for each team
df["home_goal_difference"] = (
    df["home_avg_goals_scored"] -
    df["home_avg_goals_conceded"]
)

df["away_goal_difference"] = (
    df["away_avg_goals_scored"] -
    df["away_avg_goals_conceded"]
)

# Difference between both teams
df["rating_difference"] = (
    df["home_avg_team_rating"] -
    df["away_avg_team_rating"]
)

df["goal_difference_difference"] = (
    df["home_goal_difference"] -
    df["away_goal_difference"]
)

# Average rest days
home_rest = [f"home_days_since_match_{i}" for i in range(1, 11)]
away_rest = [f"away_days_since_match_{i}" for i in range(1, 11)]

df["home_avg_rest_days"] = df[home_rest].mean(axis=1)
df["away_avg_rest_days"] = df[away_rest].mean(axis=1)

# Ratio of recent home matches
home_home = [f"home_club_recent_played_at_home_{i}" for i in range(1, 11)]
away_home = [f"away_club_recent_played_at_home_{i}" for i in range(1, 11)]

df["home_recent_home_ratio"] = df[home_home].mean(axis=1)
df["away_recent_home_ratio"] = df[away_home].mean(axis=1)

# Ratio of recent cup matches
home_cup = [f"home_club_recent_cup_game_{i}" for i in range(1, 11)]
away_cup = [f"away_club_recent_cup_game_{i}" for i in range(1, 11)]

df["home_recent_cup_ratio"] = df[home_cup].mean(axis=1)
df["away_recent_cup_ratio"] = df[away_cup].mean(axis=1)

## Feature Selection using SelectKBest

To reduce the dimensionality of the dataset and retain only the most informative features, the SelectKBest filter method is applied.

Since this is a classification problem, the ANOVA F-test (`f_classif`) is used to evaluate the relationship between each feature and the target variable. The top k highest-scoring features are then selected and used for model training.

In [44]:
# Separate features and target
X = df.drop(columns=["match_result"])
y = df["match_result"]

selector = SelectKBest(score_func=f_classif, k=50)

selector.fit(X, y)

selected_features = X.columns[selector.get_support()]

# New DataFrame 
df_selected = df[selected_features].copy()

# Goal column
df_selected["match_result"] = y

print(f"Number of selected features: {len(selected_features)}")
print("\nSelected features:")
print(selected_features.tolist())

df_selected.head()

Number of selected features: 50

Selected features:
['home_club_recent_goals_conceded_1', 'home_club_recent_team_rating_1', 'home_club_recent_team_rating_2', 'home_club_recent_team_rating_3', 'home_club_recent_team_rating_4', 'home_club_recent_team_rating_5', 'home_club_recent_team_rating_6', 'home_club_recent_team_rating_7', 'home_club_recent_team_rating_8', 'home_club_recent_team_rating_9', 'home_club_recent_team_rating_10', 'home_club_recent_opponent_rating_1', 'home_club_recent_opponent_rating_2', 'home_club_recent_opponent_rating_3', 'home_club_recent_opponent_rating_4', 'home_club_recent_opponent_rating_5', 'home_club_recent_opponent_rating_6', 'home_club_recent_opponent_rating_7', 'home_club_recent_opponent_rating_8', 'home_club_recent_opponent_rating_9', 'home_club_recent_opponent_rating_10', 'away_club_recent_team_rating_1', 'away_club_recent_team_rating_2', 'away_club_recent_team_rating_3', 'away_club_recent_team_rating_4', 'away_club_recent_team_rating_5', 'away_club_recent_

,home_club_recent_goals_conceded_1,home_club_recent_team_rating_1,home_club_recent_team_rating_2,home_club_recent_team_rating_3,home_club_recent_team_rating_4,home_club_recent_team_rating_5,home_club_recent_team_rating_6,home_club_recent_team_rating_7,home_club_recent_team_rating_8,home_club_recent_team_rating_9,...,home_avg_goals_conceded,home_avg_team_rating,away_avg_goals_scored,away_avg_goals_conceded,away_avg_team_rating,home_goal_difference,away_goal_difference,rating_difference,goal_difference_difference,match_result
0,1.0,3.856860,5.724370,4.335091,6.678853,5.478300,5.858534,3.641945,7.957243,4.326252,...,1.0,5.496371,1.6,0.9,8.561214,0.3,0.7,-3.064844,-0.4,0
1,2.0,13.668800,5.967622,9.130611,5.732981,7.804064,6.743764,6.237028,12.616250,8.334650,...,1.0,8.440839,0.8,1.4,5.287316,-0.1,-0.6,3.153522,0.5,2
2,1.0,5.736719,9.745283,5.685920,6.975000,3.864360,7.930120,4.650054,12.803284,5.094975,...,1.7,6.844712,2.1,1.1,7.620071,0.2,1.0,-0.775358,-0.8,1
3,3.0,5.998800,5.860496,8.256900,8.342183,6.163600,8.097475,5.796913,10.739525,7.261250,...,1.3,7.449923,2.4,2.0,5.818128,0.5,0.4,1.631795,0.1,0
4,2.0,6.295743,7.625358,5.320906,7.425725,4.854167,10.857700,5.756838,7.079583,5.653375,...,1.4,7.079634,1.8,1.0,5.983338,0.4,0.8,1.096297,-0.4,2


## Save Engineered Data
The engineered dataframe is saved and ready to be loaded into the modeling pipeline.

In [46]:
df_selected.to_csv("../data/engineered_data.csv", index=False)
print(f"Feature engineering complete. Data matrix shape: {df_selected.shape}")

Feature engineering complete. Data matrix shape: (110921, 51)
